# Lecture 16
## The same workflow in Polars
## Week 6 Wednesday
## Miles Chen, PhD

**How do familiar data-wrangling ideas transfer to another library?**

# Transfer the ideas, then learn the syntax

We have already inspected, filtered, cleaned, summarized, joined, and
reshaped tables in pandas. Today we will express a familiar Titanic analysis
in Polars, paying attention to the same rows, keys, and denominators.

Our main tools are `select`, `filter`, `with_columns`, and `group_by().agg()`.
We will finish with a short demonstration of lazy execution.

You do not need to memorize two complete libraries. Learn how to identify
the operation you need and read the corresponding expression.

# Why learn another library?

Polars offers an expression-based interface, parallel execution, and a lazy
query optimizer. These can be useful for composing and running data
transformations. Depending on the task and the data, Polars can potentially offer much greater performance.

Pandas remains useful, including for existing analysis code and tools that
expect pandas objects.

In [ ]:
import polars as pl

titanic = pl.read_csv("../data/titanic.csv")
print("Polars version:", pl.__version__)
print("Rows and columns:", titanic.shape)
titanic.head()

# Inspect the table

`shape` gives the row and column counts. `schema` gives column names and
data types. `null_count()` counts missing values in each column.

Polars uses `null` for missing values. We should find the same missing ages
and embarkation codes as in pandas.

In [ ]:
titanic.schema

In [ ]:
titanic.null_count()

# A familiar set of verbs

| Goal | R / dplyr | pandas | Polars |
| --- | --- | --- | --- |
| Choose columns | `select()` | `df[[...]]` | `df.select(...)` |
| Keep rows | `filter()` | `df.loc[condition]` | `df.filter(...)` |
| Create or replace a column | `mutate()` | `df["new"] = ...` | `df.with_columns(...)` |
| Sort rows | `arrange()` | `sort_values(...)` | `sort(...)` |
| Summarize groups | `group_by()` + `summarize()` | `groupby(...).agg(...)` | `group_by(...).agg(...)` |

Polars does not have pandas' labeled row index. Keep identifiers such as
`passengerid` in ordinary columns.

# Expressions describe a computation

`pl.col("age")` is an **expression** referring to a column. It is a recipe
that Polars evaluates in a context such as `select`, `filter`, or `agg`.

By contrast, `titanic["age"]` retrieves the actual column as a Series.

We can build an expression before choosing the table on which to evaluate
it. `.alias(...)` names its result.

In [ ]:
relatives_expr = pl.col("sibsp") + pl.col("parch")

print("Column expression:", type(pl.col("age")))
print("Actual column:", type(titanic["age"]))

# `select`: choose the output columns

Strings can select existing columns directly. Expressions can select and
compute new output columns. Both calls below return a DataFrame.

In [ ]:
titanic.select("passengerid", "name", "age", "fare").head()

In [ ]:
titanic.select(
    "passengerid",
    "name",
    relatives_expr.alias("relatives_aboard"),
).head()

# `filter`: keep rows where an expression is true

The syntax for combining conditions will look familiar: use `&` for
elementwise "and," `|` for "or," and parentheses around comparisons.

Select the columns we want to inspect after filtering.

In [ ]:
first_class_women = titanic.filter(
    (pl.col("pclass") == 1) & (pl.col("sex") == "female")
)
print("Passengers selected:", first_class_women.height)
first_class_women.select("passengerid", "name", "age", "survived").head()

In [ ]:
titanic.filter(pl.col("pclass").is_in([1, 2])).select(
    "passengerid", "pclass", "fare"
).head()

# `with_columns`: retain columns and add computations

`select` returns the columns you request. `with_columns` retains existing
columns and adds or replaces named columns.

These methods return new DataFrames. Assign a result to a name when you
want to use it later; this call does not change `titanic`.

`sibsp + parch` counts recorded siblings/spouses and parents/children aboard.
We call it `relatives_aboard`. It excludes the passenger themself and does
not measure every possible travel companion.

In [ ]:
passengers = titanic.with_columns(
    relatives_expr.alias("relatives_aboard")
)
passengers.select("passengerid", "sibsp", "parch", "relatives_aboard").head()

# A second step can use the newly created column

Expressions in a single `with_columns` call use that call's input table.
To refer to a column just created in the previous step, use another call.

In [ ]:
passengers = passengers.with_columns(
    (pl.col("relatives_aboard") == 0).alias("no_recorded_relatives")
)
passengers.select("passengerid", "relatives_aboard", "no_recorded_relatives").head()

# `sort`: make the requested order explicit

Sort fares from highest to lowest. If two fares are equal, use ascending
passenger ID to make the displayed order reproducible.

Polars uses `descending`; pandas uses `ascending`.

In [ ]:
passengers.select("passengerid", "name", "fare").sort(
    ["fare", "passengerid"], descending=[True, False]
).head(10)

# Your turn: translate a passenger-level analysis

Use the `passengers` table to find **third-class survivors with no recorded
relatives aboard**.

- Keep `passengerid`, `name`, `age`, `fare`, and `relatives_aboard`.
- Sort by fare from highest to lowest, breaking ties by ascending ID.
- Report the total number of selected passengers and show the first ten.
- Should a missing age exclude a passenger from this question? Explain.

In [ ]:
# Translate the filtering, selection, and sorting steps here.

# One solution

Age is not part of the selection criteria. We retain a selected passenger
even when their age was not recorded.

In [ ]:
selected_passengers = (
    passengers.filter(
        (pl.col("pclass") == 3)
        & (pl.col("survived") == 1)
        & pl.col("no_recorded_relatives")
    )
    .select("passengerid", "name", "age", "fare", "relatives_aboard")
    .sort(["fare", "passengerid"], descending=[True, False])
)
print("Selected passengers:", selected_passengers.height)
selected_passengers.head(10)

# Missing values still require a decision

`is_null()` identifies missing values; `is_not_null()` identifies recorded
values. Use these methods rather than comparing a value to `None`.

`filter` retains rows where its predicate is `True`. Both `False` and
`null` predicates are excluded. A comparison involving a missing age does
not establish that the passenger is an adult or a child.

In [ ]:
print("Missing ages:", titanic.filter(pl.col("age").is_null()).height)
print("Recorded ages:", titanic.filter(pl.col("age").is_not_null()).height)

titanic.filter(pl.col("age") >= 18).select("passengerid", "age").head()

# Group summaries use expressions too

`group_by` defines the groups; `agg` evaluates summary expressions within
each group. `.alias(...)` gives each result a clear name.

- `pl.len()` counts **rows**, including rows with null values.
- `pl.col("age").count()` counts **non-null ages**.
- `pl.col("age").mean()` averages the non-null ages.
- `pl.col("survived").mean()` gives the survival proportion among recorded
  0/1 outcomes.

The missing numeric values in this CSV are nulls. Floating-point `NaN`
values are a different case, shown in an optional example.

In [ ]:
group_summary = (
    titanic.group_by(["pclass", "sex"])
    .agg(
        pl.len().alias("n_passengers"),
        pl.col("survived").count().alias("n_survival"),
        pl.col("survived").mean().alias("survival_rate"),
        pl.col("age").count().alias("n_age"),
        pl.col("age").mean().alias("mean_age"),
    )
    .sort(["pclass", "sex"])
)
group_summary

# Read a result before adding more code

There should be six rows, representing the same class/sex groups as in
lecture 15. Sort the result explicitly; grouping alone does not promise
this display order.

For third-class women, compare `n_passengers`, `n_survival`, and `n_age`.
Which denominator belongs with each mean? Would dropping passengers with
missing ages before grouping answer the same survival question?

In [ ]:
group_summary.filter((pl.col("pclass") == 3) & (pl.col("sex") == "female"))

In [ ]:
group_summary.select(
    pl.col("n_passengers").sum().alias("total_passengers"),
    pl.col("n_survival").sum().alias("recorded_survival_outcomes"),
    pl.col("n_age").sum().alias("recorded_ages"),
)

# Compute a percentage from the summary

`n_passengers - n_age` counts passengers whose age is missing in each group.
Divide by the group's passenger count to get the fraction, then multiply
by 100 for a percentage.

We can keep the survival analysis population intact while reporting how
much age information is missing.

In [ ]:
group_summary = group_summary.with_columns(
    (100 * (pl.col("n_passengers") - pl.col("n_age")) / pl.col("n_passengers"))
    .alias("missing_age_pct")
)
group_summary

# Missing group labels: a default worth checking

Polars retains a group for null keys. To obtain this behavior in the pandas
examples from lecture 14, we used `dropna=False`.

The port summary should include the two passengers with no recorded port.
Check the total rather than assuming another library's defaults are the same.

In [ ]:
port_summary = titanic.group_by("embarked").agg(
    pl.len().alias("n_passengers")
).sort("embarked")
print("Passengers represented:", port_summary["n_passengers"].sum())
port_summary

# join tables

We will join on `pclass`, keep every group on the left,
and require one label per class on the right.

Polars spells this relationship `validate="m:1"`. As in pandas, the check
rejects repeated lookup keys but does not require all left keys to match.
Check for missing labels separately.

In [ ]:
class_labels = pl.DataFrame({
    "pclass": [1, 2, 3],
    "class_name": ["First", "Second", "Third"],
})
analysis_table = group_summary.join(
    class_labels, on="pclass", how="left", validate="m:1"
).sort(["pclass", "sex"])

print("Rows before and after:", group_summary.height, analysis_table.height)
print("Missing class names:", analysis_table["class_name"].null_count())
analysis_table

# Your turn: reproduce the analysis-ready table

Create the same `plot_table` as in lecture 15:

- Select `pclass`, `class_name`, `sex`, `n_passengers`, `n_survival`, and
  `survival_rate` from `analysis_table`.
- Sort by class and sex, and check the number of group rows and the total
  number of passengers represented.
- Explain the third-class female survival rate using its denominator.
- Identify one decision in this workflow that would remain the same in
  pandas, Polars, or R.

In [ ]:
# Create plot_table, check its counts, and explain a result.

# One solution

Six group rows represent 891 passengers. Third-class women have 72 recorded
survivors among 144 recorded outcomes: a rate of 0.50.

The library changes how we express the operations. The population,
grouping variables, key relationship, and interpretation remain the same.

In [ ]:
plot_table = analysis_table.select(
    "pclass", "class_name", "sex", "n_passengers", "n_survival", "survival_rate"
).sort(["pclass", "sex"])

print("Group rows:", plot_table.height)
print("Distinct class/sex keys:", plot_table.select("pclass", "sex").unique().height)
print("Passengers represented:", plot_table["n_passengers"].sum())
plot_table

# A brief look at lazy execution

Our DataFrame operations so far have used **eager execution**: each call
evaluates its transformation and returns a result.

With a **LazyFrame**, we build a plan for the transformations. Polars can
optimize the combined plan before computing the result with `collect()`.
`scan_csv` starts a lazy query from a CSV file. Planning can inspect file
metadata or infer a schema; it does not materialize the full result table.

We will compare first- and second-class survival using the same filter and
summaries in both approaches. We do not filter on age.

In [ ]:
eager_result = (
    titanic.filter(pl.col("pclass").is_in([1, 2]))
    .group_by(["pclass", "sex"])
    .agg(
        pl.len().alias("n_passengers"),
        pl.col("survived").count().alias("n_survival"),
        pl.col("survived").mean().alias("survival_rate"),
    )
    .sort(["pclass", "sex"])
)
eager_result

In [ ]:
lazy_query = (
    pl.scan_csv("../data/titanic.csv")
    .filter(pl.col("pclass").is_in([1, 2]))
    .group_by(["pclass", "sex"])
    .agg(
        pl.len().alias("n_passengers"),
        pl.col("survived").count().alias("n_survival"),
        pl.col("survived").mean().alias("survival_rate"),
    )
    .sort(["pclass", "sex"])
)
type(lazy_query)

# Inspect the plan, then collect the result

`explain(optimized=True)` describes the optimized plan. Look for a column
projection and the class filter near the CSV scan. Polars can avoid
materializing unused columns and pass the filter into the scan; a CSV scan
may still need to read through the file.

The exact printed plan can vary by version. Focus on the operations, not
memorizing the plan's formatting.

In [ ]:
print(lazy_query.explain(optimized=True))

In [ ]:
lazy_result = lazy_query.collect()
lazy_result

In [ ]:
from polars.testing import assert_frame_equal

assert_frame_equal(eager_result, lazy_result)
print("Eager and lazy results agree.")
print("Passengers represented:", lazy_result["n_passengers"].sum())

# What changed, and what did not?

The lazy query organizes how the computation is executed. It does not
change the intended filter, group definitions, or denominators: both
versions describe the same 400 first- and second-class passengers.

Starting with `scan_csv` gives the optimizer access to the file scan.
Calling `.lazy()` on a DataFrame is also possible, but that DataFrame has
already been loaded into memory.

We have not timed these examples. Small-data timings are not evidence that
one library will always be faster for every workflow.

# From tables to visualization

We now have the same analysis-ready structure in two libraries: one row per
class/sex group, explicit labels, a survival rate, and its denominator.

In the visualization lectures, we can map class to an axis, survival rate
to height, and sex to color. The choice of chart comes after deciding what
the rows and summaries mean. Use the table or array input supported by the
particular plotting function.

**Exit check:** What is an expression? How does `select` differ from
`with_columns`? Why are `len` and `count` sometimes different? What does
`collect()` do?

# reshaping the rates with `pivot()`

In lecture 15, `pivot` arranged one rate per class/sex group into a wide
table. Polars uses `on` for the column whose values become column labels.

We already have one value per class/sex key, so no aggregation is needed.
Keep the fuller `analysis_table` to retain denominators.

In [ ]:
survival_wide = group_summary.pivot(
    on="sex", index="pclass", values="survival_rate"
).sort("pclass")
survival_wide

# `unpivot` is the wide-to-long operation

Here `on` names the measurement columns to gather, and `index` names the
identifier columns to keep. These are ordinary columns; Polars has no
pandas-style row index to reset.

The `sex` values come from column labels. The numeric `pclass` values remain
in their original identifier column.

In [ ]:
survival_long = survival_wide.unpivot(
    on=["female", "male"],
    index="pclass",
    variable_name="sex",
    value_name="survival_rate",
).sort(["pclass", "sex"])
survival_long

# conditions must account for unknown values

Suppose we want an age label using 18 as the threshold. Handle missing ages
first. Otherwise, an `otherwise(pl.lit("child"))` branch would incorrectly classify
a missing age as evidence that a passenger was a child.

In expression contexts such as `then`, a bare string is interpreted as a
column name. Use `pl.lit("adult")` to supply a literal string. Numeric
literals can be used directly in arithmetic and comparisons, as above.

In [ ]:
age_labeled = titanic.with_columns(
    pl.when(pl.col("age").is_null())
    .then(pl.lit("not recorded"))
    .when(pl.col("age") >= 18)
    .then(pl.lit("adult"))
    .otherwise(pl.lit("child"))
    .alias("age_status")
)
age_labeled.group_by("age_status").agg(
    pl.len().alias("n_passengers")
).sort("age_status")

# `null` and `NaN` are different

A null is a missing value. `NaN` is a special floating-point value that can
arise from numeric operations. Polars does not treat `NaN` as null:
`count` includes it, and a mean can be `NaN` even after nulls are skipped.

Inspect the values and their source before choosing `fill_null`, `fill_nan`,
or a filtering rule. They address different conditions.

In [ ]:
numeric_example = pl.DataFrame({"x": [1.0, None, float("nan")]})
numeric_example.select(
    pl.len().alias("n_rows"),
    pl.col("x").count().alias("n_non_null"),
    pl.col("x").null_count().alias("n_null"),
    pl.col("x").is_nan().sum().alias("n_nan"),
    pl.col("x").mean().alias("mean_x"),
)

# join defaults can differ across libraries

Polars does not match null join keys by default. Pandas can match missing
keys to missing keys. When a match between missing keys has an appropriate
meaning, Polars provides `nulls_equal=True`.

Our class-label join uses complete keys, so this difference does not affect
the result. Missing keys, repeated keys, and unmatched keys are still
separate things to check.

# Optional: streaming execution

A lazy query can request `collect(engine="streaming")`. Supported parts of
the query can process data in batches, which can reduce working memory.
Support depends on the operations and version, and some work may fall back
to the in-memory engine.

`collect` still returns a materialized DataFrame. Streaming is not a promise
that an arbitrarily large final result will fit in memory. This small
example illustrates the API, not a memory or speed benefit.

In [ ]:
streaming_result = lazy_query.collect(engine="streaming")
assert_frame_equal(eager_result, streaming_result)
streaming_result

# References

- [Polars: expressions and contexts](https://docs.pola.rs/user-guide/concepts/expressions-and-contexts/)
- [Polars: aggregation](https://docs.pola.rs/user-guide/expressions/aggregation/)
- [Polars: missing data](https://docs.pola.rs/user-guide/expressions/missing-data/)
- [Polars: joins](https://docs.pola.rs/user-guide/transformations/joins/)
- [Polars: lazy execution](https://docs.pola.rs/user-guide/concepts/lazy-api/)
- [Polars: streaming](https://docs.pola.rs/user-guide/concepts/streaming/)